# Continuous-Space Diffusion Trajectory Training

End-to-end pipeline:
1. Data generation — RRT* in continuous 2D space, obstacles rasterized to 64×64 occupancy maps
2. Dataset loading & normalization
3. Model construction — ConditionalUnet1D with CNN map encoder
4. Training — DDPM with FiLM conditioning
5. Inference & visualization

In [ ]:
import sys
sys.path.insert(0, '.')   # run from repo root

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches

## 1. Data Generation

Generates N samples, each with:
- `start`, `goal`  : 2D points in [0, 8] × [0, 8]
- `paths`          : RRT*-interpolated trajectory  (50, 2)
- `map`            : 64×64 binary occupancy image (1 = obstacle)

Saved to `dataset/train_continuous.npy`.

In [ ]:
from data_generator.data_generator import DataGenerator2D

BOUNDS        = [(0, 8), (0, 8)]
NUM_SAMPLES   = 2000       # increase for better performance
MAP_RES       = 64         # must match config cnn_config['image_size']
INTERP_POINTS = 50         # must match config horizon
OUTFILE       = 'dataset/train_continuous.npy'

gen = DataGenerator2D(
    bounds=BOUNDS,
    num_samples=NUM_SAMPLES,
    max_obstacles=5,
    map_resolution=MAP_RES,
    interp_points=INTERP_POINTS,
    outfile=OUTFILE,
)
gen.generate()

### Quick sanity-check: visualize a few samples

In [ ]:
data = np.load(OUTFILE, allow_pickle=True).item()
print('Keys    :', list(data.keys()))
print('start   :', data['start'].shape)
print('goal    :', data['goal'].shape)
print('paths   :', data['paths'].shape)
print('map     :', data['map'].shape)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, ax in enumerate(axes):
    ax.imshow(data['map'][i], origin='lower', cmap='gray_r',
              extent=[0, 8, 0, 8])
    path = data['paths'][i]          # (50, 2)
    ax.plot(path[:, 0], path[:, 1], 'b-', linewidth=1.5, label='path')
    ax.scatter(*data['start'][i], c='g', s=80, zorder=5, label='start')
    ax.scatter(*data['goal'][i],  c='r', s=80, zorder=5, label='goal')
    ax.set_title(f'Sample {i}')
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

## 2. Dataset & Normalization

In [ ]:
from core.datasets.plane_dataset_embed import PlanePlanningDataSets

dataset = PlanePlanningDataSets(OUTFILE)
print(f'Dataset size : {len(dataset)}')

sample = dataset[0]
print('sample keys  :', list(sample.keys()))
print('trajectory   :', sample['sample'].shape)   # (T, 2), normalized
print('map          :', sample['map'].shape)       # (1, 64, 64)
print('env (s+g)    :', sample['env'].shape)       # (4,)

## 3. Model Construction

- **CNN** encodes 64×64 occupancy map → 128-dim latent
- **MLP** encodes [start, goal] → 64-dim latent
- **ConditionalUnet1D** denoises trajectories conditioned on the concatenated latent (192-dim)
  via FiLM modulation

In [ ]:
from config.plane_continuous import PlaneContinuousConfig
from core.networks.embedUnet import ConditionalUnet1D
from core.diffusion.builder import build_noise_scheduler_from_config

config     = PlaneContinuousConfig()
config_dict = config.to_dict()

# global_cond_dim = cnn_output_dim + mlp_embed_dim = 128 + 64 = 192
cnn_output_dim = config.network_config['cnn_config']['output_dim']
mlp_embed_dim  = config.network_config['mlp_config']['embed_dim']

net = ConditionalUnet1D(
    input_dim       = config.action_dim,
    global_cond_dim = cnn_output_dim + mlp_embed_dim,   # 192
    network_config  = config.network_config,
    is_cnn          = config.is_CNN,
)

noise_scheduler = build_noise_scheduler_from_config(config_dict)

total_params = sum(p.numel() for p in net.parameters())
print(f'Total parameters: {total_params:,}')

# --- quick shape test ---
dummy_action = torch.randn(2, config.horizon, config.action_dim)
dummy_map    = torch.randn(2, 1, 64, 64)
dummy_env    = torch.randn(2, 4)
dummy_t      = torch.zeros(2).long()
out = net(dummy_action, dummy_t, dummy_map, dummy_env)
print(f'Forward pass output shape: {out.shape}')   # should be (2, 50, 2)

## 4. Training

In [ ]:
from core.trainer.plane_diffusion_trainer_embed import PlaneDiffusionTrainer

NUM_EPOCHS      = 5000
SAVE_CKPT_EPOCH = 50    # save checkpoint every N epochs

trainer = PlaneDiffusionTrainer(
    net     = net,
    dataset = dataset,
    config  = config,
)

train_losses = trainer.train(num_epochs=NUM_EPOCHS, save_ckpt_epoch=SAVE_CKPT_EPOCH)

# save final checkpoint
trainer.save_checkpoint('ckpt_continuous_final.ckpt')
print('Training complete.')

### Loss curve

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_losses)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training Loss')
plt.grid(True)
plt.tight_layout()
plt.show()

## 5. Inference & Visualization

Run reverse diffusion to generate a trajectory given a start, goal, and occupancy map.

In [ ]:
from core.diffusion.policy import PlaneDiffusionPolicy

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

policy = PlaneDiffusionPolicy(
    model           = net,
    noise_scheduler = noise_scheduler,
    config          = config_dict,
    device          = DEVICE,
)

# pick a test sample from the dataset
test_idx = 10
raw_data  = np.load(OUTFILE, allow_pickle=True).item()

obs_dict = {
    'env': np.concatenate([raw_data['start'][test_idx],
                           raw_data['goal'][test_idx]]),   # (4,)
    'map': raw_data['map'][test_idx][None],                # (1, 64, 64)
}

initial_action = torch.randn(1, config.horizon, config.action_dim, device=DEVICE)
pred_path, _ = policy.predict_action(obs_dict, initial_action)
print('Predicted path shape:', pred_path.shape)  # (50, 2)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(raw_data['map'][test_idx], origin='lower', cmap='gray_r',
          extent=[0, 8, 0, 8])

gt_path = raw_data['paths'][test_idx]
ax.plot(gt_path[:, 0],   gt_path[:, 1],   'b--', linewidth=1.5, label='ground truth')
ax.plot(pred_path[:, 0], pred_path[:, 1], 'r-',  linewidth=2.0, label='diffusion pred')

ax.scatter(*raw_data['start'][test_idx], c='g', s=100, zorder=6, label='start')
ax.scatter(*raw_data['goal'][test_idx],  c='r', s=100, zorder=6, label='goal')

ax.set_xlim(0, 8)
ax.set_ylim(0, 8)
ax.set_title(f'Sample {test_idx}')
ax.legend()
plt.tight_layout()
plt.show()